# Inter-Rater Agreement Analysis — axSpA Radiographic Classification Study

**Cel:** obliczenie zgodności między M.P. i P.G. PRZED konsensusem — dla manuskryptu Diagnostics MDPI.

**Co dostajesz na końcu:**
- Cohen's kappa (unweighted, linear, quadratic weighted)
- 95% CI (bootstrap 2000 iteracji)
- Kappa per cohort (Łódź vs Końskie)
- Confusion matrix między readerami
- Heatmap wizualizacyjna
- **Gotowy tekst do wklejenia do Methods §2.4**

## Krok 1 — Instalacja pakietów

In [ ]:
!pip install -q openpyxl pandas scikit-learn matplotlib seaborn numpy

## Krok 2 — Wgraj plik Excel

Kliknij "Choose files" i wybierz swój Excel z ocenami M.P. i P.G. dla wszystkich 827 przypadków.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
excel_files = [f for f in uploaded.keys() if f.endswith(('.xlsx', '.xls'))]

if not excel_files:
    raise FileNotFoundError('Nie wgrano pliku Excel')

EXCEL_FILE = excel_files[0]
print(f'\nWgrano: {EXCEL_FILE} ({os.path.getsize(EXCEL_FILE)} bytes)')

## Krok 3 — Konfiguracja

**DOSTOSUJ do nazw kolumn w Twoim pliku Excel!**

Jeśli nie wiesz jak są nazwane, uruchom tę komórkę — pokaże dostępne kolumny.

In [ ]:
import pandas as pd

# Podgląd struktury pliku
df_preview = pd.read_excel(EXCEL_FILE)
print(f'Wiersze: {len(df_preview)}')
print(f'Kolumny: {list(df_preview.columns)}')
print(f'\nPierwsze 5 wierszy:')
print(df_preview.head())

**Teraz ustaw nazwy kolumn** (widziałeś je powyżej):

In [ ]:
# ================ CONFIG — DOSTOSUJ ================
COL_READER_MP = 'M.P.'       # <- kolumna z ocenami M.P. (10 lat doświadczenia)
COL_READER_PG = 'P.G.'       # <- kolumna z ocenami P.G. (15 lat doświadczenia)
COL_COHORT    = 'Cohort'      # <- kolumna z ośrodkiem (Lodz/Konskie) — opcjonalne
SHEET_NAME    = 0             # 0 = pierwszy arkusz albo nazwa np. 'Sheet1'

# Jeśli oceny są tekstowe (Normal/Osteo/…), ustaw mapowanie.
# Jeśli są już liczbami 0-3, zostaw None.
CATEGORY_MAP = None
# Przykład:
# CATEGORY_MAP = {'Normal': 0, 'Osteophytes': 1, 'Parasyndesmophytes': 2, 'Syndesmophytes': 3}

# ================ NIE ZMIENIAJ PONIŻEJ ================
import pandas as pd
import numpy as np
from sklearn.metrics import cohen_kappa_score, confusion_matrix

df = pd.read_excel(EXCEL_FILE, sheet_name=SHEET_NAME)

# Walidacja kolumn
for col in [COL_READER_MP, COL_READER_PG]:
    if col not in df.columns:
        raise ValueError(f"Kolumna '{col}' nie istnieje. Dostępne: {list(df.columns)}")

# Wyciągnij oceny
mp = df[COL_READER_MP].copy()
pg = df[COL_READER_PG].copy()

# Mapuj tekst → liczby jeśli trzeba
if CATEGORY_MAP is not None:
    mp = mp.map(CATEGORY_MAP)
    pg = pg.map(CATEGORY_MAP)

# Usuń wiersze z brakującymi ocenami
mask = mp.notna() & pg.notna()
mp = mp[mask].astype(int).values
pg = pg[mask].astype(int).values

n_total = len(mp)
n_dropped = len(df) - n_total

print(f'✅ Wczytano {n_total} par ocen do analizy')
if n_dropped > 0:
    print(f'⚠️  Pominięto {n_dropped} wierszy z brakującymi ocenami')

# Sprawdź czy oceny są w prawidłowym zakresie 0-3
print(f'\nZakres ocen M.P.: {mp.min()} do {mp.max()}')
print(f'Zakres ocen P.G.: {pg.min()} do {pg.max()}')

if mp.max() > 3 or pg.max() > 3:
    print('⚠️  Uwaga: niektóre oceny > 3. Sprawdź skalę!')

## Krok 4 — Oblicz podstawowe wskaźniki zgodności

Ta komórka liczy:
- **Exact agreement** — ile ocen jest identycznych
- **Within-1-grade agreement** — ile jest ±1 stopień
- **Major discrepancies** — rozbieżności o ≥2 stopnie

In [ ]:
# Podstawowe wskaźniki zgodności
n_agree = int(np.sum(mp == pg))
n_disagree = n_total - n_agree
pct_agree = n_agree / n_total * 100

diff = np.abs(mp - pg)
n_diff_1 = int(np.sum(diff == 1))
n_diff_2 = int(np.sum(diff == 2))
n_diff_3 = int(np.sum(diff == 3))
n_within_1 = int(np.sum(diff <= 1))
pct_within_1 = n_within_1 / n_total * 100

n_major = int(np.sum(diff >= 2))
pct_major = n_major / n_total * 100

print('=' * 65)
print(f'ZGODNOŚĆ MIĘDZY READERAMI (n={n_total})')
print('=' * 65)
print(f'Exact agreement:         {n_agree:>4} / {n_total} = {pct_agree:>5.1f}%')
print(f'Within-1-grade:          {n_within_1:>4} / {n_total} = {pct_within_1:>5.1f}%')
print(f'Discrepancies:           {n_disagree:>4} / {n_total} = {100-pct_agree:>5.1f}%')
print(f'   ↳ 1-grade diff:       {n_diff_1:>4}')
print(f'   ↳ 2-grade diff:       {n_diff_2:>4}')
print(f'   ↳ 3-grade diff:       {n_diff_3:>4}')
print(f'Major (≥2 grade):        {n_major:>4} / {n_total} = {pct_major:>5.1f}%')

## Krok 5 — Cohen's kappa (główna metryka)

Liczymy 3 wersje kappa + bootstrap 95% CI.

**Do manuskryptu użyjemy `quadratic-weighted` kappa** — właściwe dla ordinal grading (kary za większe rozbieżności są kwadratowe).

In [ ]:
from sklearn.metrics import cohen_kappa_score

# Trzy warianty kappa
kappa_unweighted = cohen_kappa_score(mp, pg)
kappa_linear = cohen_kappa_score(mp, pg, weights='linear')
kappa_quadratic = cohen_kappa_score(mp, pg, weights='quadratic')

# Bootstrap 95% CI dla quadratic-weighted (main metric)
np.random.seed(42)
n_boot = 2000
kappa_boot = []
for _ in range(n_boot):
    idx = np.random.choice(n_total, size=n_total, replace=True)
    try:
        k = cohen_kappa_score(mp[idx], pg[idx], weights='quadratic')
        kappa_boot.append(k)
    except:
        pass
kappa_boot = np.array(kappa_boot)
ci_low, ci_high = np.percentile(kappa_boot, [2.5, 97.5])

# Interpretacja wg Landis & Koch (1977)
def interpret_kappa(k):
    if k >= 0.81: return 'almost perfect'
    elif k >= 0.61: return 'substantial'
    elif k >= 0.41: return 'moderate'
    elif k >= 0.21: return 'fair'
    elif k >= 0.01: return 'slight'
    else: return 'poor'

interp = interpret_kappa(kappa_quadratic)

print('=' * 65)
print('COHEN\'S KAPPA')
print('=' * 65)
print(f'Unweighted:                    κ = {kappa_unweighted:.3f}')
print(f'Linear weighted:               κ = {kappa_linear:.3f}')
print(f'Quadratic weighted (główna):   κ = {kappa_quadratic:.3f}')
print(f'                              95% CI: [{ci_low:.3f}, {ci_high:.3f}]')
print(f'Interpretacja (Landis & Koch): {interp}')

## Krok 6 — Kappa per cohort (Łódź vs Końskie)

Jeśli masz kolumnę z ośrodkiem, obliczamy kappa osobno — silny argument o generalizowalności.

In [ ]:
kappa_by_cohort = {}

if COL_COHORT in df.columns:
    for cohort in df[COL_COHORT].dropna().unique():
        cohort_mask = df[COL_COHORT] == cohort
        combined_mask = cohort_mask & mask
        mp_c = df.loc[combined_mask, COL_READER_MP].values
        pg_c = df.loc[combined_mask, COL_READER_PG].values

        if CATEGORY_MAP:
            mp_c = np.array([CATEGORY_MAP.get(x, x) for x in mp_c])
            pg_c = np.array([CATEGORY_MAP.get(x, x) for x in pg_c])
        mp_c = mp_c.astype(int)
        pg_c = pg_c.astype(int)

        if len(mp_c) > 5:
            k_cohort = cohen_kappa_score(mp_c, pg_c, weights='quadratic')
            pct_c = np.sum(mp_c == pg_c) / len(mp_c) * 100
            kappa_by_cohort[cohort] = {
                'n': len(mp_c),
                'kappa_quadratic': k_cohort,
                'pct_agree': pct_c
            }
            print(f'{cohort} (n={len(mp_c)}):')
            print(f'   Exact agreement: {pct_c:.1f}%')
            print(f'   Quadratic-weighted κ: {k_cohort:.3f} ({interpret_kappa(k_cohort)})')
            print()
else:
    print(f'⚠️  Brak kolumny "{COL_COHORT}" — pomijam analizę per cohort')

## Krok 7 — Confusion matrix między readerami

Wizualna heatmapa pokazuje **gdzie** readerzy się nie zgadzają — najczęściej między Grade 1 (osteo) i Grade 2 (para).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

class_names = ['Normal\n(0)', 'Osteophytes\n(1)', 'Parasyndes.\n(2)', 'Syndesmoph.\n(3)']
cm = confusion_matrix(mp, pg, labels=[0, 1, 2, 3])

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Panel A: raw counts
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title('Confusion Matrix — Raw counts', fontsize=13, fontweight='bold')
axes[0].set_xlabel('P.G. (secondary reader, 15 years)', fontsize=11)
axes[0].set_ylabel('M.P. (primary reader, 10 years)', fontsize=11)

# Panel B: normalized (row-wise)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
sns.heatmap(cm_norm, annot=True, fmt='.1f', cmap='RdYlGn',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], cbar_kws={'label': '% of M.P. row'})
axes[1].set_title('Confusion Matrix — Row-normalized (%)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('P.G. (secondary reader, 15 years)', fontsize=11)
axes[1].set_ylabel('M.P. (primary reader, 10 years)', fontsize=11)

plt.tight_layout()
plt.savefig('/content/interrater_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✅ Zapisano: /content/interrater_confusion_matrix.png')

## Krok 8 — Zapisz wszystkie wyniki

Wyniki zapisujemy do JSON i CSV — przydadzą się do supplementary materials.

In [ ]:
import json

# Zbierz wszystko
results = {
    'sample_size': {
        'n_total': n_total,
        'n_dropped_due_to_missing': n_dropped,
    },
    'agreement': {
        'n_exact_agreement': n_agree,
        'pct_exact_agreement': round(pct_agree, 2),
        'n_within_1_grade': n_within_1,
        'pct_within_1_grade': round(pct_within_1, 2),
        'n_discrepancy_1_grade': n_diff_1,
        'n_discrepancy_2_grade': n_diff_2,
        'n_discrepancy_3_grade': n_diff_3,
        'n_major_discrepancy_2plus': n_major,
        'pct_major_discrepancy_2plus': round(pct_major, 2),
    },
    'cohens_kappa': {
        'unweighted': round(kappa_unweighted, 4),
        'linear_weighted': round(kappa_linear, 4),
        'quadratic_weighted': round(kappa_quadratic, 4),
        'quadratic_weighted_ci95_low': round(ci_low, 4),
        'quadratic_weighted_ci95_high': round(ci_high, 4),
        'interpretation_landis_koch': interp,
        'bootstrap_iterations': n_boot,
    },
    'cohens_kappa_per_cohort': kappa_by_cohort,
    'confusion_matrix': cm.tolist(),
    'class_labels': ['Normal (0)', 'Osteophytes (1)', 'Parasyndesmophytes (2)', 'Syndesmophytes (3)'],
}

with open('/content/interrater_results.json', 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

# CSV z confusion matrix
cm_df = pd.DataFrame(cm,
                     index=[f'M.P.={c}' for c in ['Normal', 'Osteo', 'Para', 'Syndes']],
                     columns=[f'P.G.={c}' for c in ['Normal', 'Osteo', 'Para', 'Syndes']])
cm_df.to_csv('/content/interrater_confusion_matrix.csv')

print('✅ Zapisano:')
print('  /content/interrater_results.json')
print('  /content/interrater_confusion_matrix.csv')
print('  /content/interrater_confusion_matrix.png')

## Krok 9 — GOTOWY TEKST DO MANUSKRYPTU 📋

**To jest to co masz mi podesłać** — skopiuj i wklej w wiadomości do mnie.

In [ ]:
print('=' * 70)
print('SKOPIUJ TĘ SEKCJĘ I PODEŚLIJ ASYSTENTOWI')
print('=' * 70)
print()

# Formatowanie zależy czy mamy dane per cohort
cohort_text = ''
if kappa_by_cohort:
    parts = []
    for cohort, stats in kappa_by_cohort.items():
        parts.append(f"κ={stats['kappa_quadratic']:.3f} in the {cohort} cohort (n={stats['n']})")
    cohort_text = f" Kappa was consistent across cohorts ({'; '.join(parts)})."

manuscript_text = f"""Prior to consensus resolution, the two readers agreed on {n_agree} of {n_total} radiographs ({pct_agree:.1f}%), with {n_disagree} cases ({100-pct_agree:.1f}%) requiring joint review. Discrepancies were within one ordinal grade in {n_within_1}/{n_total} cases ({pct_within_1:.1f}%); {n_major} cases ({pct_major:.1f}%) showed disagreement of two or more grades. Quadratic-weighted Cohen's kappa for initial reader agreement was {kappa_quadratic:.3f} (95% CI {ci_low:.3f}-{ci_high:.3f}), corresponding to '{interp}' agreement according to Landis and Koch.{cohort_text}"""

print(manuscript_text)
print()
print('=' * 70)
print('DODATKOWO — kluczowe liczby:')
print('=' * 70)
print(f'  Exact agreement:        {pct_agree:.1f}%')
print(f'  Within-1-grade:         {pct_within_1:.1f}%')
print(f'  Major discrepancy (2+): {pct_major:.1f}%')
print(f'  Quadratic-weighted κ:   {kappa_quadratic:.3f} [{ci_low:.3f}, {ci_high:.3f}]')
print(f'  Interpretation:         {interp}')

## Krok 10 — Pobierz wyniki (opcjonalnie)

Pobierz wszystkie zapisane pliki — możesz je dołączyć jako supplementary materials do submisji.

In [ ]:
from google.colab import files
import shutil

# Zbierz wszystko do zip
shutil.make_archive('/content/interrater_analysis', 'zip', '/content', base_dir='.')

# Zaproponuj pobranie kluczowych plików
files.download('/content/interrater_results.json')
files.download('/content/interrater_confusion_matrix.png')
files.download('/content/interrater_confusion_matrix.csv')